# Compare email and web notices

Source differences can affect both annotation effort and extraction errors. Compare text lengths, sentence counts, and date forms before choosing model input lengths or feature windows.

The default dataset is synthetic. Its plots check the workflow and do not support research conclusions. Change `DATASET` in the setup cell after collecting and preparing a reviewed corpus.

In [ ]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from smartnotice.demo import run_demo
from smartnotice.io import read_jsonl
from smartnotice.schemas import Notice

%matplotlib inline

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), (
    "Open this notebook from the repository root or notebooks/"
)

DATASET = "demo"  # Options: "demo", "snapshot", "research".
assert DATASET in {"demo", "snapshot", "research"}
if DATASET == "demo":
    run_demo(ROOT)
    NOTICE_PATH = ROOT / "data/interim/demo/notices.jsonl"
    SPLIT_PATH = ROOT / "data/processed/demo/splits.jsonl"
    ANNOTATION_PATH = ROOT / "data/annotations/examples/events.jsonl"
    PAIR_PATH = ROOT / "data/processed/demo/applicability.jsonl"
elif DATASET == "snapshot":
    NOTICE_PATH = ROOT / "data/catalogs/hcmus-focused-100/notices.jsonl"
else:
    NOTICE_PATH = ROOT / "data/interim/notices.jsonl"
    SPLIT_PATH = ROOT / "data/processed/splits.jsonl"
    ANNOTATION_PATH = ROOT / "data/annotations/adjudicated.jsonl"
    PAIR_PATH = ROOT / "data/processed/applicability.jsonl"

FIGURES = ROOT / "reports/generated" / DATASET / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
NOTICE_SHA256 = hashlib.sha256(NOTICE_PATH.read_bytes()).hexdigest()
notices = [Notice.model_validate(row) for row in read_jsonl(NOTICE_PATH)]
if hashlib.sha256(NOTICE_PATH.read_bytes()).hexdigest() != NOTICE_SHA256:
    raise ValueError("Notice input changed during loading; rerun setup")
assert notices, "Prepare at least one notice before analysis"
frame = pd.DataFrame([n.model_dump(mode="json") for n in notices])
print(f"Dataset: {DATASET}; documents: {len(notices)}; synthetic: {int(frame.synthetic.sum())}")
if DATASET == "research":
    assert not frame.synthetic.any(), "Exclude synthetic fixtures from research measurements"

In [ ]:
frame["characters"] = frame.text.str.len()
frame["sentences_count"] = frame.sentences.map(len)
frame["numeric_dates"] = frame.text.str.count(r"\b\d{1,2}[/-]\d{1,2}(?:[/-]\d{4})?\b")
frame["relative_date_mentions"] = frame.text.str.count(r"(?i)tuần này|tuần sau|ngày mai|hôm nay")
frame["table_lines"] = frame.text.map(
    lambda text: sum("; " in line and ": " in line for line in text.splitlines())
)
display(
    frame.groupby("source_type")[
        ["characters", "sentences_count", "numeric_dates", "relative_date_mentions", "table_lines"]
    ].agg(["count", "median", "min", "max"])
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sources = sorted(frame.source_type.unique())
axes[0].boxplot(
    [frame.loc[frame.source_type == source, "characters"] for source in sources],
    tick_labels=sources,
)
axes[1].boxplot(
    [frame.loc[frame.source_type == source, "sentences_count"] for source in sources],
    tick_labels=sources,
)
axes[0].set(title="Notice length by source", ylabel="Unicode characters")
axes[1].set(title="Sentence count by source", ylabel="Sentences")
fig.tight_layout()
fig.savefig(FIGURES / "eda_source_lengths.png", dpi=160)
plt.show()
plt.close("all")

## Inspect reference dates by month and source

Count documents with a non-null `reference_date` in calendar-month bins (`YYYY-MM`), split by `source_type` (`email`/`web`). Include months with zero documents between the earliest and latest dates. The coverage table reports missing dates separately; collection dates and unverified publication candidates never fill them.

Choose `snapshot` in the setup cell to inspect the shared HCMUS-focused-100 corpus. If every reference date is missing, the figure reports that no monthly distribution is available. CSV tables and the figure contain aggregate counts only. The JSON record identifies the input hash, dataset, date range, and software versions for this run.


In [ ]:
import hashlib
import json
import platform

import matplotlib
from matplotlib.ticker import MaxNLocator
from smartnotice.quality import reference_date_counts

if hashlib.sha256(NOTICE_PATH.read_bytes()).hexdigest() != NOTICE_SHA256:
    raise ValueError("Notice input changed since loading; rerun setup before exporting")

monthly, coverage = reference_date_counts(notices)
display(coverage)
display(monthly)
assert int(monthly.to_numpy().sum()) == int(coverage.dated.sum())
assert int(coverage.total.sum()) == len(notices)

fig, axis = plt.subplots(figsize=(11, 4.8))
if monthly.empty:
    axis.text(
        0.5,
        0.5,
        "No reference dates available; see missing-date counts",
        ha="center",
        va="center",
        transform=axis.transAxes,
    )
    axis.set_xticks([])
    axis.set_ylim(0, 1)
else:
    monthly.set_axis(monthly.index.astype(str)).plot.bar(stacked=True, ax=axis, width=1.0, rot=45)
    axis.set_xlim(-0.5, len(monthly) - 0.5)
    axis.legend(title="Source")
    # Limit label density while retaining every monthly bin.
    step = max(1, (len(monthly) + 17) // 18)
    positions = list(range(0, len(monthly), step))
    axis.set_xticks(positions, [str(monthly.index[i]) for i in positions])
    for label in axis.get_xticklabels():
        label.set_ha("right")
axis.yaxis.set_major_locator(MaxNLocator(integer=True))
axis.set(
    title=f"Reference dates by month and source ({DATASET})\n"
    f"Dated: {int(coverage.dated.sum())}; missing: {int(coverage.missing.sum())}",
    xlabel="Reference month (YYYY-MM)",
    ylabel="Documents (count)",
)
fig.tight_layout()
fig.savefig(FIGURES / "eda_reference_dates.png", dpi=160)
monthly.to_csv(FIGURES / "eda_reference_dates_monthly.csv", encoding="utf-8")
coverage.to_csv(FIGURES / "eda_reference_dates_coverage.csv", encoding="utf-8")
record = {
    "dataset": DATASET,
    "notice_path": NOTICE_PATH.relative_to(ROOT).as_posix(),
    "notice_sha256": NOTICE_SHA256,
    "documents": len(notices),
    "synthetic_documents": sum(n.synthetic for n in notices),
    "dated_documents": int(coverage.dated.sum()),
    "missing_documents": int(coverage.missing.sum()),
    "first_month": str(monthly.index[0]) if not monthly.empty else None,
    "last_month": str(monthly.index[-1]) if not monthly.empty else None,
    "bin": "calendar month",
    "source_field": "source_type",
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
}
(FIGURES / "eda_reference_dates_run.json").write_text(
    json.dumps(record, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
plt.show()
plt.close(fig)

## Inspect date expression coverage

Regex counts describe surface forms. They do not establish extracted dates or normalized-date accuracy. Sentence segmentation uses punctuation and line boundaries, so abbreviations and table formatting need inspection.

In [ ]:
counts = frame.groupby("source_type")[
    ["numeric_dates", "relative_date_mentions", "table_lines"]
].sum()
display(counts)
axis = counts.plot.bar(figsize=(9, 3.8), rot=0, title="Date mentions and table-like lines")
axis.set_ylabel("Count")
axis.figure.tight_layout()
axis.figure.savefig(FIGURES / "eda_date_forms.png", dpi=160)
plt.show()
plt.close("all")

## Inspect a prepared example

The preview below contains prepared text. Complete privacy review before exporting an executed notebook from a real corpus.

In [ ]:
sample = notices[0]
print(sample.doc_id, sample.source_type, sample.reference_date)
for sentence in sample.sentences:
    print(f"{sentence.sentence_id} [{sentence.start}:{sentence.end}] {sentence.text}")

## Write an interpretation

For a reviewed research corpus, connect observed source differences to annotation time, date ambiguity, input length, and candidate relation distances. State sample sizes and link saved figures. Avoid generalizing from the synthetic examples.